**Task-Specific Classes:**
- to tackle binary classification using BERT, we can use the BertForSequenceClassification class provided by the transformers lib. (Just a BERT model plus a classification head added on top of it). need to do is specify the checkpoint, number of output units, and optionally the output datatype (we use 16-bit floats to fit on small GPUs)

In [1]:
from transformers import BertForSequenceClassification

In [2]:
import torch
import torch.nn as nn

In [3]:
from transformers import AutoTokenizer
from datasets import load_dataset

In [4]:
data = load_dataset("imdb")
split = data['train'].train_test_split(train_size=0.8, seed=42)
imdb_train_set, imdb_val_set = split['train'], split['test']
imdb_test_set = data['test']

In [5]:
imdb_train_set

Dataset({
    features: ['text', 'label'],
    num_rows: 20000
})

In [6]:
z = imdb_train_set.filter(lambda x: x['label']==1)

In [7]:
print(z['text'][2])

Two years ago I watched "The Matador" in cinema and I loved everything about this movie. Obviously, I was totally under impression of Pierce Brosan's magnificent role. Yesterday, I caught this movie again on TV so I looked at it a bit deeper. Now, I can say with certain that this movie isn't that special but you just gotta' love it because of one man. <br /><br />Brosnan lifts its grade up in my opinion with amazing performance of Julian Noble, tired hit-man who has no friends. Soon Julian meets Danny Wright (Greg Kinnear) in Mexico City, man who's got bad luck: his son died in accident, his job isn't going that well and he's not sure that he can keep his wife Bean (Hope Davis).<br /><br />I always liked movies like this; crime movie with big touch of humor. Mostly that humor comes from Brosnan as he tells jokes about dwarfs with big d.... or one of my favorite lines in this movie: "I look like a Bangkok hooker on a Sunday morning, after the navy's left town." Brosnan says it with his 

In [8]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [9]:
torch.manual_seed(42)

In [10]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")

In [65]:
bert_for_binary_clf = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2, dtype=torch.float16
).to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tx lib: has many task-specific classes based on various pretrained models, such as BertForQuestionAnswering or RobertaForSequenceClassification. Can also use AutoModelForSequenceClassification to let the library pick the right class....

In [25]:
encoding = bert_tokenizer(["This was a great movie"], return_tensors="pt").to(device)

In [26]:
with torch.no_grad():
    output = bert_for_binary_clf(**encoding)
output.logits

tensor([[0.1162, 0.0618]], device='mps:0', dtype=torch.float16)

In [27]:
output.logits.shape

torch.Size([1, 2])

In [28]:
torch.softmax(output.logits, dim=1)

tensor([[0.5137, 0.4863]], device='mps:0', dtype=torch.float16)

model returns a ModelOutput object containing the logits. 

if you pass labels when calling this model (or any other model from the transformers lib), it also computes the loss and returns it in the ModelOutput object. For example:

In [29]:
with torch.no_grad():
    output = bert_for_binary_clf(
        **encoding,
        labels=torch.tensor([1], device=device)
    )
output.loss

tensor(0.7207, device='mps:0', dtype=torch.float16)

**The Trainer API:**
- The Trainer API lets u finetune your own dataset with minimal boilerplate code. 
- Trainer API works with dataset objects not dataloaders (hence why it can di batching, shuffling, and padding). It expects the datasets to contain tokenized text though not strings, so we must take care of tokenization. This can be done using the dataset map method

In [87]:
bert_for_binary_clf = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2, dtype=torch.float16
).to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [88]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [89]:
def tokenize_batch(batch):
    return bert_tokenizer(batch['text'], truncation=True, max_length=512)

In [90]:
imdb_train_set

Dataset({
    features: ['text', 'label'],
    num_rows: 20000
})

In [91]:
tok_imdb_train_set = imdb_train_set.map(tokenize_batch, batched=True)
tok_imdb_val_set = imdb_val_set.map(tokenize_batch, batched=True)
tok_imdb_test_set = imdb_test_set.map(tokenize_batch, batched=True)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

The tokenize_batch method tokenizes the given batch of reviews, and the resulting fields are added to each instance by the map method. This includes fields such as token_ids and attention_mask which the model expects....

To evaluate our model, we can write a simple function that takes an object with 2 attributes: label_ids and predictions...

In [92]:
# for example a function to compute the accuracy:
def compute_accuracy(pred):
    # takes an object with label_ids and predictions arguments
    return {"accuracy": (pred.label_ids == pred.predictions.argmax(-1)).mean()}

# alternatively, could use the metrics provided by the huggingface evaluate library

Next, we mist specify our training config in a TrainingArguments object

In [93]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

In [94]:
# freeze the bert model's params
for param in bert_for_binary_clf.bert.parameters():
    param.requires_grad = False

# classifier params should stay trainable
for param in bert_for_binary_clf.classifier.parameters():
    param.requires_grad = True

verify that this has been done:

In [95]:
for name, p in bert_for_binary_clf.named_parameters():
    if p.requires_grad:
        print(name)

classifier.weight
classifier.bias


only the classifiers params are trainable now

In [96]:
train_args = TrainingArguments(
    output_dir="my_imdb_model", # the output dir where the model predictions and checkpoints will be written
    num_train_epochs=2, # the total number of training epochs to perform
    per_device_train_batch_size=32, # the batch size per device. Global batch size computed as: per_device_train_batch_size*num_of_devices
    per_device_eval_batch_size=32,
    eval_strategy = "epoch", # eval strategy to adopt during training. 'no' - no eval, 'steps' - eval is done every eval steps, 'epoch' - eval at end of each epoch
    logging_strategy="epoch", # logging strategy to adopt during training
    save_strategy="epoch", # the checkpoint save strategy to adopt during training
    load_best_model_at_end=True, # whether or not to load the best model at the end of training
    metric_for_best_model="accuracy", # metric to use to consider best model, if left unspecified, loss is used
    report_to="none" # list of integrations
    )

Lastly, create a Trainer object and pass it the model, along with the training arguments, the training and validation sets, the evaluation function, plus a data collator which will take care of padding. plus a data collator which will take care of padding. Finally, we call the trainer's train method. 

In [97]:
trainer = Trainer(
    bert_for_binary_clf,
    train_args,
    train_dataset=tok_imdb_train_set,
    eval_dataset=tok_imdb_val_set,
    compute_metrics=compute_accuracy,
    data_collator=DataCollatorWithPadding(bert_tokenizer)
)

In [98]:
trainer.train()

/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,60.167600,67.562500,0.553800
2,56.990300,65.875000,0.556800


/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=1250, training_loss=58.578903125, metrics={'train_runtime': 3397.9513, 'train_samples_per_second': 11.772, 'train_steps_per_second': 0.368, 'total_flos': 1.05228799925088e+16, 'train_loss': 58.578903125, 'epoch': 2.0})

test preds on the test set:

In [51]:
enc = bert_tokenizer(["Bad negative!"], return_tensors="pt").to(device)

with torch.no_grad():
    o = bert_for_binary_clf(**enc)
o.logits

tensor([[0.6665, 5.7812]], device='mps:0', dtype=torch.float16)

In [52]:
torch.softmax(o.logits, dim=-1)

tensor([[0.0060, 0.9941]], device='mps:0', dtype=torch.float16)

In [58]:
torch.argmax(o.logits).item()

1

In [49]:
tok_imdb_test_set

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})

In [61]:
acc = []
for i in range(tok_imdb_test_set.num_rows):
    d = tok_imdb_test_set[i]
    with torch.no_grad():
        enc = bert_tokenizer(d['text'], return_tensors='pt', truncation=True, max_length=200).to(device)
        o = bert_for_binary_clf(**enc)
        o = torch.argmax(o.logits, dim=-1).item()
        pred = int(o) == int(d['label'])
        acc.append(pred)

In [64]:
sum(acc)/len(acc)

0.518

accuracy is still quite bad - probably something to do with truncating the inputs...

Notes:
- By default for most HF models:
  - classification: CrossEntropyLoss
  - regression: MSELoss
- Loss computation can be overridden as follows:

In [ ]:
# can override the way the loss is computed for training as follows:
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)

        logits = outputs.logits

        # YOUR CRITERION
        criterion = torch.nn.BCEWithLogitsLoss(reduction="sum")
        loss = criterion(logits.squeeze(), labels.float())

        # can also add regularization
        # L1 regularization
        l1_lambda = 1e-5
        l1_norm = sum(p.abs().sum() for p in model.parameters())
        loss = loss + l1_lambda * l1_norm

        # L2 regularization (manual, not weight decay)
        l2_lambda = 1e-4
        l2_norm = sum(p.pow(2).sum() for p in model.parameters())
        loss = loss + l2_lambda * l2_norm

        return (loss, outputs) if return_outputs else loss

# then use:
trainer = CustomTrainer(

)


# optimizer and scheduler:
# simplest way to use is to pass them directly to the Trainer
# the trainer class has an optimizers argument - which is a tuple containing the optimizer and the scheduler to use - will default to an instance of AdamW and
# a scheduler given by get_linear_schedule_with_warmup() controlled by args
# optimizers: the training_arguments has an optim and learning_rate arguments
# the trainer takes an optimizers tuple for custom pytorch optimizers
# default scheduler is linear decay with warmup ("linear") - controlled via lr_scheduler_type in TrainingArguments